# AnalystLab Africa — Week 2: Superstore Data Cleaning & Preparation
**Goal:** Inspect, clean, and transform the Superstore dataset in Python (pandas), then export a Tableau-ready CSV with calculated fields already added.

This notebook covers:
1. Import & inspect the dataset
2. Check missing values
3. Check & remove duplicates
4. Fix data types
5. Basic cleaning & transformations
6. Add calculated columns (Profit Margin, Order Year/Month, Shipping Duration)
7. Export clean CSV for Tableau


## 1. Import Libraries & Load Dataset

In [1]:
import pandas as pd
import numpy as np

# Superstore CSV is encoded in latin1, not utf-8
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

print("Shape:", df.shape)
df.head()

Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 2. Review Dataset Structure

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,9994.0,NaN,NaN,NaN,4997.5,2885.163629,1.0,2499.25,4997.5,7495.75,9994.0
Order ID,9994,5009,CA-2017-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9994,1237,9/5/2016,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,9994,1334,12/16/2015,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,9994,4,Standard Class,5968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9994,793,WB-21850,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9994,793,William Brown,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9994,3,Consumer,5191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,9994,1,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9994,531,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Check for Missing Values

In [4]:
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print("No missing values found.")

No missing values found.


## 4. Check & Remove Duplicate Records

In [5]:
dupe_count = df.duplicated().sum()
print(f"Duplicate rows found: {dupe_count}")

df = df.drop_duplicates()
print("Shape after dedup:", df.shape)

Duplicate rows found: 0
Shape after dedup: (9994, 21)


## 5. Correct Data Types
`Order Date` and `Ship Date` are stored as text — convert to datetime. `Postal Code` should be treated as a category/string, not a number (it's an ID, not a quantity).

In [6]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)  # keep leading zeros (e.g. Northeast zips)

df.dtypes

Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code                 str
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

## 6. Basic Cleaning
Strip whitespace from text columns and standardize category labels.

In [7]:
text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Sanity check unique values in key categorical fields
for col in ['Segment', 'Region', 'Category', 'Sub-Category', 'Ship Mode']:
    print(col, ':', df[col].unique())

Segment : <StringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str
Region : <StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str
Category : <StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str
Sub-Category : <StringArray>
[  'Bookcases',      'Chairs',      'Labels',      'Tables',     'Storage',
 'Furnishings',         'Art',      'Phones',     'Binders',  'Appliances',
       'Paper', 'Accessories',   'Envelopes',   'Fasteners',    'Supplies',
    'Machines',     'Copiers']
Length: 17, dtype: str
Ship Mode : <StringArray>
['Second Class', 'Standard Class', 'First Class', 'Same Day']
Length: 4, dtype: str


/tmp/ipykernel_507/3558794110.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns


## 7. Add Calculated Columns
These make it faster to build KPIs and time-based visuals directly in Tableau (though you can also do these as Tableau calculated fields).

In [8]:
# Profit Margin = Profit / Sales
df['Profit Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], 0)

# Shipping duration in days
df['Shipping Duration'] = (df['Ship Date'] - df['Order Date']).dt.days

# Order Year / Month / Month-Year for time trend analysis
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')
df['Order Year-Month'] = df['Order Date'].dt.to_period('M').astype(str)

# Flag loss-making orders (useful for a risk-focused visual)
df['Is Loss'] = df['Profit'] < 0

df[['Sales','Profit','Profit Margin','Shipping Duration','Order Year-Month','Is Loss']].head()

,Sales,Profit,Profit Margin,Shipping Duration,Order Year-Month,Is Loss
0,261.9600,41.9136,0.1600,3,2016-11,False
1,731.9400,219.5820,0.3000,3,2016-11,False
2,14.6200,6.8714,0.4700,4,2016-06,False
3,957.5775,-383.0310,-0.4000,7,2015-10,True
4,22.3680,2.5164,0.1125,7,2015-10,False


## 8. Quick Sanity Checks (Outliers & Ranges)

In [9]:
print("Date range:", df['Order Date'].min(), "to", df['Order Date'].max())
print("Negative sales rows:", (df['Sales'] < 0).sum())
print("Loss-making orders:", df['Is Loss'].sum(), f"({df['Is Loss'].mean():.1%} of all orders)")
print("Max discount:", df['Discount'].max(), "| Min discount:", df['Discount'].min())

Date range: 2014-01-03 00:00:00 to 2017-12-30 00:00:00
Negative sales rows: 0
Loss-making orders: 1871 (18.7% of all orders)
Max discount: 0.8 | Min discount: 0.0


## 9. Export Cleaned Dataset for Tableau

In [10]:
output_path = "Superstore_Cleaned.csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')  # utf-8-sig plays nicely with Tableau on Windows/Mac
print(f"Cleaned file saved to: {output_path}")
print("Final shape:", df.shape)

Cleaned file saved to: Superstore_Cleaned.csv
Final shape: (9994, 28)


## 10. Next Steps in Tableau
1. Connect Tableau to `Superstore_Cleaned.csv`.
2. Confirm field types on import: `Postal Code` → String, `Order Date`/`Ship Date` → Date, `Is Loss` → Boolean.
3. Build KPI calculated fields (see the separate Tableau formula reference).
4. Use `Order Year-Month` or `Order Date` (continuous) for the trend line chart.
5. Use `Region`/`State` for the map visual.
